# 🦴 OsteoVision — EXP-002f 再訓練ノートブック

**目的:** 中間回旋角データセット拡張（720 → 1296枚）による YOLO11s-pose 再訓練

**EXP-002f 変更点:**
- `rots = [-30, -15, 0, 15, 30]` → `[-30, -15, -10, -5, 0, 5, 10, 15, 30]`
- 総画像数: 720 → 1296 枚
- モデル: YOLOv8n-pose → YOLO11s-pose
- 目標: 全回旋角で 80%+ 検出率

**実行前に:** 上のメニューから `ランタイム → ランタイムのタイプを変更 → T4 GPU` を選択してください。

---

| ステップ | 内容 |
|---|---|
| STEP 1 | ライブラリのインストール |
| STEP 2 | Google Drive マウント |
| STEP 3 | リポジトリのクローン |
| STEP 4 | EXP-002f データセット生成 + 訓練（train_exp002f.py 実行） |
| STEP 5 | 訓練済みモデルを Drive に保存 |
| STEP 6 | 結果確認 |

## STEP 1: ライブラリのインストール

In [ ]:
!pip install ultralytics opencv-python-headless numpy scipy pydicom pandas tqdm -q
print('✅ インストール完了')

## STEP 2: Google Drive をマウント

訓練済みモデル (`best.pt`) と訓練ログを Drive に保存します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/OsteoVision/EXP002f'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ 保存先: {SAVE_DIR}')

## STEP 3: リポジトリのクローン

GitHub から最新コードを取得します。データ生成・訓練スクリプトが含まれています。

In [ ]:
import os

REPO_URL = 'https://github.com/tomosoko/OsteoVision.git'
REPO_DIR = '/content/OsteoVision'

if os.path.exists(REPO_DIR):
    print('リポジトリ既存 — git pull で更新します')
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

# OsteoSynth を Python パスに追加
import sys
sys.path.insert(0, f'{REPO_DIR}/OsteoSynth')

print('✅ リポジトリ準備完了')
!git -C {REPO_DIR} log --oneline -5

## STEP 4: EXP-002f データセット生成 + 訓練

`train_exp002f.py` を実行します。このスクリプトは:
1. `yolo_pose_factory.py` を使って 1296 枚の合成DRRを生成
2. YOLO11s-pose を 150 エポック訓練（GPU使用で約60〜90分）

訓練完了後、結果は `/content/runs/pose/osteo_exp002f/` に保存されます。

In [ ]:
import subprocess
import sys

os.chdir(f'{REPO_DIR}/OsteoSynth')
print(f'作業ディレクトリ: {os.getcwd()}')

# GPU 確認
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nデバイス: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️ GPU が見つかりません。ランタイムのタイプを T4 GPU に変更してください。')

In [ ]:
# train_exp002f.py を実行
# 完了まで T4 GPU で 60〜90 分かかります
!python train_exp002f.py

## STEP 5: 訓練済みモデルを Google Drive に保存

In [ ]:
import shutil
import glob

# runs ディレクトリは OsteoSynth の親（=リポジトリルート）を基準に作成される
run_dir = f'{REPO_DIR}/runs/pose/osteo_exp002f'
best_pt = f'{run_dir}/weights/best.pt'
last_pt = f'{run_dir}/weights/last.pt'

if os.path.exists(best_pt):
    shutil.copy(best_pt, f'{SAVE_DIR}/best.pt')
    print(f'✅ best.pt を保存: {SAVE_DIR}/best.pt')
else:
    print(f'⚠️ best.pt が見つかりません: {best_pt}')
    print('runs/ ディレクトリ内を確認します...')
    for f in glob.glob(f'{REPO_DIR}/runs/**/*.pt', recursive=True):
        print(f'  {f}')

# 訓練ログ・曲線も保存
for fname in ['results.png', 'results.csv']:
    src = f'{run_dir}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{SAVE_DIR}/{fname}')
        print(f'✅ {fname} を保存')

print(f'\n保存先: {SAVE_DIR}')
print('次のステップ: best.pt を dicom-viewer-prototype-api/best.pt に配置して検出率を評価')

## STEP 6: 訓練結果の確認

In [ ]:
from IPython.display import Image, display
import os

result_img = f'{run_dir}/results.png'
if os.path.exists(result_img):
    display(Image(result_img))
else:
    print('results.png が見つかりません')

# 最終メトリクスを results.csv から表示
import pandas as pd
results_csv = f'{run_dir}/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    print('\n=== 最終エポック結果 ===')
    for col in ['metrics/mAP50(P)', 'metrics/mAP50(B)', 'metrics/precision(P)', 'metrics/recall(P)']:
        if col in last:
            print(f'  {col}: {last[col]:.4f}')
else:
    print('results.csv が見つかりません')

## 訓練完了後の手順

1. Google Drive から `best.pt` をダウンロード
2. `dicom-viewer-prototype-api/best.pt` に配置
3. 中角度回旋での検出率を評価:
   ```bash
   cd OsteoSynth
   python validate_synth_drr.py
   ```
4. 目標: 全回旋角（±5°, ±10° 含む）で 80%+ 検出
5. 達成後: EXP-002d キャリブレーション係数を再計算